# FX Technical Strategy Research Lab

**一份 Notebook + 一份原始数据，复现全部 Technical Team 研究。**

适用范围：9 种外币相对 USD；**20 套技术规则**（另附 2 个方向基准）；不构建多货币 Portfolio。原始数据来自同目录的 `FX_Technical_PIT_Dataset(6).xlsx`（2000-01-03—2026-05-29）。

**执行方式（四种）：**

1. **Daily Continuous** — 每日收盘形成信号，下一收盘执行，仓位持续至信号改变，包含周末暴露。
2. **Daily Weekflat** — 同样每日更新，但周五收盘后空仓，排除周五到周一的收益。
3. **Weekly Flat** — 上周最后一个收盘价的信号，本周第一个收盘价开仓，最后一个收盘价平仓（通常为周一到周五，四个日收益区间）。
4. **Fixed 5D** — 每隔 5 个价格观测日决定一次方向，下一个收盘价执行，持有 5 个完整的 close-to-close 日收益区间。

研究范围：独立 P&L、Sharpe/回撤/换仓、货币分类、原有宏观 Regime 事后归因、技术状态、年度/历史阶段稳定性、完整交易台账、持仓路径与极端亏损、同步年度区块 Bootstrap、全部 180 条曲线、可离线导出的 HTML 交互图。

**统一限制：** 使用 USD per foreign currency 的收盘价；`+1` 做多外币、`−1` 做多 USD、`0` 空仓。**零成本、无 carry/融资/实际点差**；全历史事后描述，不代表未来 OOS 交易业绩。仅有 Close，Donchian 是收盘价突破，不是真实日内 High/Low 突破。宏观 Regime 只做归因，**不训练宏观预测模型**。

> 按顺序执行 **Run All**。Notebook 仅依赖同目录的 Excel；全部导出物运行后新建在 `outputs/`，无需额外的预计算 CSV 或旧 Python 脚本。

## 0. 环境与可调整参数

安装 Python 3.10+、Jupyter、NumPy、Pandas、Plotly（另建议 Matplotlib）。如缺少包，可以在自己的终端运行：`python -m pip install numpy pandas plotly matplotlib jupyter`。Excel 的读取由下文的 Python 标准库完成，不依赖 Excel 引擎。可选的 `ipywidgets` 不影响重算。

In [ ]:
from pathlib import Path
from datetime import datetime, date, timedelta
from collections import defaultdict
from bisect import bisect_right
import xml.etree.ElementTree as ET
import zipfile, csv, json, re, math, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# 改动研究条件时，优先只修改这里
SOURCE_XLSX = Path('FX_Technical_PIT_Dataset(6).xlsx')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
CURRENCIES = ('EUR', 'GBP', 'AUD', 'NZD', 'JPY', 'CHF', 'CAD', 'NOK', 'SEK')
MODES = ('Daily Continuous', 'Daily Weekflat', 'Weekly Flat', 'Fixed 5D')
MAIN_MODE = 'Daily Continuous'  # 可改为另外三种
GROUP_SCENARIO = 'Base'
RISK_STRESS_VIX, RISK_STRESS_HY = 25.0, 6.0
RISK_CALM_VIX, RISK_CALM_HY = 20.0, 5.0
ZERO_COST_BP = 0.0  # 用户确认暂时不扣交易成本
BOOTSTRAP_DRAWS = 500
RANDOM_SEED = 20261021
PLOT_SAMPLE = 'month_end'    # month_end / daily
assert SOURCE_XLSX.exists(), f'请把 Notebook 与 {SOURCE_XLSX.name} 放在同一文件夹'
print('Data:', SOURCE_XLSX.resolve())

## 1. 直接解析原始 Excel

仅使用 `PIT_Daily_Core` 的日收盘价、`PIT_Monthly` 的已有全球宏观快照；其他 sheet 原样保留在 Excel 中，Notebook 不修改源文件。月度全球字段在 9 个 pair 重复，这里只读取 EUR 那一份，随后应用给全部货币对。

In [ ]:
NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
XL_EPOCH = datetime(1899, 12, 30)
def excelrows(z,sheetnum,cols=None):
    fname=f'xl/worksheets/sheet{sheetnum}.xml'
    with z.open(fname) as f:
        for ev, el in ET.iterparse(f, events=('end',)):
            if el.tag != NS+'row':continue
            values = {}
            for c in el.findall(NS+'c'):
                m=re.match(r'([A-Z]+)',c.get('r',''))
                if not m:continue
                col=m.group(1)
                if cols is not None and col not in cols:continue
                t=c.get('t','')
                v=c.find(NS+'v')
                if t=='inlineStr':
                    values[col]=''.join(q.text or '' for q in c.iter(NS+'t'))
                elif t=='s':
                    raise ValueError('Unexpected shared strings; extend parser if format changes')
                elif v is None or not v.text:
                    values[col]=None
                elif t in ('str','e'):
                    values[col]=v.text
                else:
                    try: values[col]=float(v.text)
                    except ValueError:values[col]=v.text
            yield values
            el.clear()

def date_from_xl(x):
    return (XL_EPOCH+timedelta(days=int(x))).date().isoformat()

def read_source(path):
    points=defaultdict(lambda:{'date':[], 'px':[],'ret_src':[], 'core_ready':[]})
    monthly=defaultdict(list)
    with zipfile.ZipFile(path) as z:
        for k,row in enumerate(excelrows(z,6,{'B','C','D','E','F','N'})):
            if k==0:continue
            c=row.get('D')
            if c not in CURRENCIES:continue
            if row.get('E') is None:continue
            points[c]['date'].append(date_from_xl(row['B']))
            points[c]['px'].append(float(row['E']))
            points[c]['ret_src'].append(float(row['F']) if row.get('F') is not None else np.nan)
            points[c]['core_ready'].append(int(row['N'] or 0))
        for k,row in enumerate(excelrows(z,4,{'B','E','J','P','Q','R','S','T','U','V','W'})):
            if k==0:continue
            c=row.get('E')
            if c not in CURRENCIES:continue
            monthly[c].append({'date':date_from_xl(row['B']), 'spot':row.get('J'),
                    'rsi14':row.get('P'), 'sma50gap':row.get('Q'),
                    'sma200gap':row.get('R'), 'trend50_200':row.get('S'),
                    'macdhist':row.get('T'),'bbz20':row.get('U'),
                    'rv21':row.get('V'),'rv63':row.get('W')})
    for c in CURRENCIES:
        for field in ('px','ret_src','core_ready'):points[c][field]=np.array(points[c][field],float)
        dates=points[c]['date']
        if len(dates)!=len(set(dates)) or dates!=sorted(dates):raise ValueError(f'{c}: repeated or unsorted dates')
    return points,monthly

In [ ]:
source_daily, _source_monthend_indicators = read_source(str(SOURCE_XLSX))

# 从工作簿 PIT_Monthly 原始宏观列直接重建 Regime，不读取旧 CSV。
MACRO_XLSX_COLS = {'date':'B','currency':'E','vix_level':'AI','vix_pctile_5y':'AK',
                   'us_hy_oas':'AP','hy_oas_pctile_5y':'AR','us_2y':'AW',
                   'us_10y':'AX','wti_level':'BA','dxy_level':'BC','move_level':'AL'}
monthly_raw=[]
with zipfile.ZipFile(SOURCE_XLSX) as workbook_zip:
    for idx, row in enumerate(excelrows(workbook_zip, 4, set(MACRO_XLSX_COLS.values()))):
        if idx == 0 or row.get('E') != 'EUR':
            continue
        record = {key:row.get(col) for key,col in MACRO_XLSX_COLS.items()}
        record['date'] = pd.Timestamp(date_from_xl(record['date']))
        monthly_raw.append(record)
macro_monthly = pd.DataFrame(monthly_raw).sort_values('date').reset_index(drop=True)
for col in MACRO_XLSX_COLS:
    if col not in ('date','currency'):
        macro_monthly[col] = pd.to_numeric(macro_monthly[col], errors='coerce')

source_audit=[]
for pair in CURRENCIES:
    info=source_daily[pair]
    src_r=np.asarray(info['ret_src'],dtype=float)
    calc_r=np.r_[np.nan,np.diff(np.log(info['px']))]
    error=np.nanmax(np.abs(src_r-calc_r))
    source_audit.append({'pair':pair,'observations':len(info['px']),'first':info['date'][0],
                         'last':info['date'][-1],'log_return_max_abs_diff':error})
source_audit=pd.DataFrame(source_audit)
assert set(source_audit.observations)=={6890}, '数据覆盖与之前研究不同，请核查原始文件'
assert source_audit.log_return_max_abs_diff.max() < 1e-9
print('Data quality checks passed. Daily total:',source_audit.observations.sum(),
      '| monthly global snapshots:',len(macro_monthly))
display(source_audit)

## 2. 指标公式与原先 20 套规则

保留之前的 **RSI（反转和动量）、MA、Bollinger（均值回归和突破）、收盘价 Donchian、MACD、时间序列动量、等权/强度投票、固定阈值技术状态策略**。统一先生成 `−1 / 0 / +1` 信号，再独立应用不同执行方式。

In [ ]:
def sma(x,n):
    x=np.asarray(x,float)
    out=np.full(len(x),np.nan)
    valid=np.isfinite(x)
    sumx=np.cumsum(np.where(valid,x,0.0))
    cnt=np.cumsum(valid.astype(int))
    if len(x)>=n:
        s=sumx[n-1:] - np.concatenate(([0.0],sumx[:-n]))
        count=cnt[n-1:] - np.concatenate(([0],cnt[:-n]))
        out[n-1:]=np.where(count==n,s/n,np.nan)
    return out

def rolling_std(x,n,ddof=1):
    x=np.asarray(x,float)
    mean=sma(x,n)
    sq=sma(x*x,n)
    return np.sqrt(np.maximum((sq-mean*mean)*n/(n-ddof),0.0))

def ema(x,n):
    x=np.asarray(x,float)
    out=np.full(len(x),np.nan)
    a=2.0/(n+1)
    seeded=False
    for i,v in enumerate(x):
        if not np.isfinite(v):continue
        if not seeded:out[i]=v;seeded=True
        else:out[i]=a*v+(1-a)*out[i-1]
    return out

def rsi_wilder(p,n):
    """Wilder RSI, seeded from first observed gain/loss (source-workbook convention)."""
    diff=np.diff(p)
    up=np.maximum(diff,0.0)
    down=np.maximum(-diff,0.0)
    out=np.full(len(p),np.nan)
    if len(p)<=n:return out
    ag=float(up[0]);al=float(down[0])
    def rsival(g,l):
        return 50.0 if g==0 and l==0 else (100.0 if l==0 else 100.0-100.0/(1.0+g/l))
    for i in range(2,len(p)):
        ag=((n-1)*ag+up[i-1])/n
        al=((n-1)*al+down[i-1])/n
        if i>=n:out[i]=rsival(ag,al)
    return out

def sign_strict(x):
    arr=np.zeros(len(x),dtype=np.int8)
    arr[np.isfinite(x)&(x>0)]=1
    arr[np.isfinite(x)&(x<0)]=-1
    return arr

def bb_z(p,n):
    a=sma(p,n)
    b=rolling_std(p,n,ddof=1)
    return np.divide(p-a,b,out=np.full(len(p),np.nan),where=np.isfinite(b)&(b>1e-12))

def event_osc_reversion(r,low=30.0,high=70.0,center=50.0):
    s=np.zeros(len(r),np.int8)
    position=0
    for i in range(1,len(r)):
        v=r[i];last=r[i-1]
        if not np.isfinite(v) or not np.isfinite(last):continue
        if last<low<=v:position=1
        elif last>high>=v:position=-1
        elif position==1 and v>=center:position=0
        elif position==-1 and v<=center:position=0
        s[i]=position
    return s

def bb_strategy(z,reversion):
    s=np.zeros(len(z),np.int8)
    position=0
    for i,v in enumerate(z):
        if not np.isfinite(v):continue
        if v>=2:position=-1 if reversion else 1
        elif v<=-2:position=1 if reversion else -1
        elif position==1 and (v>=0 if reversion else v<=0):position=0
        elif position==-1 and (v<=0 if reversion else v>=0):position=0
        s[i]=position
    return s

def donchian(p,n):
    s=np.zeros(len(p),np.int8)
    midpoint=np.full(len(p),np.nan)
    pos=0
    for i in range(n,len(p)):
        prev=p[i-n:i]
        h=prev.max();l=prev.min();midpoint[i]=(h+l)/2
        if p[i]>h:pos=1
        elif p[i]<l:pos=-1
        elif pos==1 and p[i]<midpoint[i]:pos=0
        elif pos==-1 and p[i]>midpoint[i]:pos=0
        s[i]=pos
    return s,midpoint

def rolling_trend_efficiency(logp,n):
    ret=np.diff(logp,prepend=np.nan)
    absret=np.abs(ret)
    total=sma(absret,n)*n
    net=np.full(len(logp),np.nan);net[n:]=np.abs(logp[n:]-logp[:-n])
    return np.divide(net,total,out=np.full(len(logp),np.nan),where=np.isfinite(total)&(total>0))

def define_rules(p):
    p=np.array(p,float);n=len(p)
    logp=np.log(p)
    logr=np.full(n,np.nan);logr[1:]=np.diff(logp)
    vol21=rolling_std(logr,21)*np.sqrt(252)
    vol63=rolling_std(logr,63)*np.sqrt(252)
    eff63=rolling_trend_efficiency(logp,63)
    out={}
    cache={}
    for window in [7,14,20]:
        r=rsi_wilder(p,window);cache[f'rsi{window}']=r
        out[f'RSI_MR_{window}']=event_osc_reversion(r)
        mom=np.zeros(n,np.int8)
        mom[np.isfinite(r)&(r>60)]=1
        mom[np.isfinite(r)&(r<40)]=-1
        out[f'RSI_MOM_{window}']=mom
    for fast,slow in [(10,30),(20,60)]:
        a=sma(p,fast);b=sma(p,slow)
        gap=a/b-1
        out[f'MA_{fast}_{slow}']=sign_strict(gap)
        cache[f'magap_{fast}_{slow}']=gap
    for window in [10,20]:
        z=bb_z(p,window)
        cache[f'bbz{window}']=z
        out[f'BB_MR_{window}']=bb_strategy(z,True)
        out[f'BB_BO_{window}']=bb_strategy(z,False)
    for window in [10,20]:
        ds,mid=donchian(p,window)
        out[f'DONCHIAN_BO_{window}']=ds
        cache[f'donmid{window}']=mid
    macd=ema(p,12)-ema(p,26)
    signal=ema(macd,9)
    histpct=(macd-signal)/p
    cache['histpct']=histpct
    out['MACD_12_26_9']=sign_strict(histpct)
    for window in [21,63]:
        v=np.full(n,np.nan);v[window:]=logp[window:]-logp[:-window]
        out[f'TSMOM_{window}']=sign_strict(v)
    voter_names=['RSI_MOM_14','MA_20_60','DONCHIAN_BO_20','MACD_12_26_9']
    votes=np.stack([out[k].astype(float) for k in voter_names],axis=0)
    out['VOTE_EQUAL_4']=sign_strict(votes.sum(axis=0))
    # Static, deterministic conviction weights; no fitted model and no use of future returns.
    r=cache['rsi14']
    w_rsi=np.clip(np.abs(r-50)/25,0,1)
    s20=np.maximum(rolling_std(logr,21)*np.sqrt(20),1e-7)
    w_ma=np.clip(np.abs(cache['magap_20_60'])/s20,0,1)
    w_don=np.clip(np.abs(logp-np.log(np.maximum(cache['donmid20'],1e-10)))/(s20*2),0,1)
    w_mac=np.clip(np.abs(histpct)/(np.maximum(rolling_std(logr,21),1e-7)*1.5),0,1)
    weights=np.nan_to_num(np.stack([w_rsi,w_ma,w_don,w_mac]),nan=0.0)
    w_sum=(votes*weights).sum(axis=0)
    w_norm=w_sum/np.maximum(weights.sum(axis=0),1e-10)
    out['VOTE_STRENGTH_4']=np.where(w_norm>0.2,1,np.where(w_norm<-0.2,-1,0)).astype(np.int8)
    # Ex-ante regime thresholds: train/validation only, determined separately by currency.
    return out,{'rsi14':cache['rsi14'],'ma50gap':p/sma(p,50)-1,
                'ma200gap':p/sma(p,200)-1,'bbz20':cache['bbz20'],
                'macdhist':histpct,'eff63':eff63,'vol21':vol21,
                'vol63':vol63,'logr':logr}

In [ ]:
RULES={
'RSI_MR_7':('RSI','Mean reversion','7d Wilder RSI crosses back above 30 -> Long FX / below 70 -> Long USD; exit RSI 50'),
'RSI_MR_14':('RSI','Mean reversion','14d RSI crossing above 30 / below 70; exit 50'),
'RSI_MR_20':('RSI','Mean reversion','20d RSI crossing above 30 / below 70; exit 50'),
'RSI_MOM_7':('RSI','Momentum','7d RSI >60 Long FX, <40 Long USD, else flat'),
'RSI_MOM_14':('RSI','Momentum','14d RSI >60 Long FX, <40 Long USD, else flat'),
'RSI_MOM_20':('RSI','Momentum','20d RSI >60 Long FX, <40 Long USD, else flat'),
'MA_10_30':('Moving average','Trend following','SMA10 > SMA30 Long FX; SMA10 < SMA30 Long USD'),
'MA_20_60':('Moving average','Trend following','SMA20 > SMA60 Long FX; SMA20 < SMA60 Long USD'),
'BB_MR_10':('Bollinger','Mean reversion','Z=(P-SMA10)/SD10; Z>=2 Long USD; Z<=-2 Long FX; exit at Z crosses 0'),
'BB_MR_20':('Bollinger','Mean reversion','Z=(P-SMA20)/SD20; Z>=2 Long USD; Z<=-2 Long FX; exit at Z crosses 0'),
'BB_BO_10':('Bollinger','Breakout','Z>=2 Long FX; Z<=-2 Long USD; exit when Z crosses 0'),
'BB_BO_20':('Bollinger','Breakout','Z>=2 Long FX; Z<=-2 Long USD; exit when Z crosses 0'),
'DONCHIAN_BO_10':('Donchian','Breakout','Close > prior 10 closes high -> Long FX, below prior 10 closes low -> Long USD; exit channel midpoint'),
'DONCHIAN_BO_20':('Donchian','Breakout','Close > prior 20 closes high -> Long FX, below prior 20 closes low -> Long USD; exit midpoint'),
'MACD_12_26_9':('MACD','Trend following','EMA(12)-EMA(26) minus 9d signal EMA histogram sign determines position'),
'TSMOM_21':('Time-series momentum','Momentum','21d log return sign Long FX/Long USD'),
'TSMOM_63':('Time-series momentum','Momentum','63d log return sign Long FX/Long USD'),
'VOTE_EQUAL_4':('Signal voting','Ensemble','RSI MOM14 + MA20/60 + Donchian20 + MACD histogram; equal votes, tie flat'),
'VOTE_STRENGTH_4':('Signal voting','Ensemble','The same four signals with deterministic 0..1 strength weights, cutoff +/-0.2'),
'REGIME_RULE_FIXED':('Regime switch','Adaptive rule','Prior 63-day trend efficiency >0.40 -> MA20/60; <0.20 -> RSI MR14; else flat. Fixed thresholds, no fit'),
'BUY_HOLD_LONG_FX':('Benchmark','Benchmark','Always +1 foreign currency, -1 USD'),
'BUY_HOLD_LONG_USD':('Benchmark','Benchmark','Always -1 foreign currency, +1 USD')}
STRATEGIES = [name for name in RULES if name not in ('BUY_HOLD_LONG_FX','BUY_HOLD_LONG_USD')]
BENCHMARKS = ('BUY_HOLD_LONG_FX', 'BUY_HOLD_LONG_USD')
ALL_RULES = STRATEGIES + list(BENCHMARKS)
assert len(STRATEGIES)==20 and len(ALL_RULES)==22

def complete_signals(prices):
    rules, features = define_rules(prices)
    eff = features['eff63']
    fixed = np.zeros(len(prices), dtype=np.int8)
    hi = np.isfinite(eff) & (eff > .40)
    lo = np.isfinite(eff) & (eff < .20)
    fixed[hi] = rules['MA_20_60'][hi]
    fixed[lo] = rules['RSI_MR_14'][lo]
    rules['REGIME_RULE_FIXED'] = fixed
    rules['BUY_HOLD_LONG_FX'] = np.ones(len(prices),dtype=np.int8)
    rules['BUY_HOLD_LONG_USD'] = -np.ones(len(prices),dtype=np.int8)
    assert set(rules)==set(ALL_RULES), set(rules)^set(ALL_RULES)
    return rules, features

specifications=pd.DataFrame([
    {'strategy':key,'indicator':value[0],'hypothesis':value[1],'rule':value[2]}
    for key,value in RULES.items()
])
display(specifications)

## 3. 宏观与技术状态（只做事后分析）

宏观维度：VIX/HY OAS 的 `Stress / Calm / Intermediate`；五年百分位的风险状态；DXY 过去 3 个月方向；美国 2Y 利率变化；WTI 方向；9 个主观标注的宏观历史阶段。状态标签只在当时**已观察到的最近一次月末快照**上采样。月末值确切发布时间及历史数据 vintage 尚未逐一审计。历史事件分期**不能用于事前信号**。

In [ ]:
HISTORICAL_ERAS = [
 ('E1','2000-01-01','2003-12-31','Tech bust / easing'),
 ('E2','2004-01-01','2007-12-31','Global expansion / carry'),
 ('E3','2008-01-01','2009-12-31','Global financial crisis'),
 ('E4','2010-01-01','2012-12-31','Euro area sovereign stress'),
 ('E5','2013-01-01','2015-12-31','Taper and dollar/commodity divergence'),
 ('E6','2016-01-01','2019-12-31','US normalization / Brexit / trade'),
 ('E7','2020-01-01','2021-12-31','Pandemic shock and easing'),
 ('E8','2022-01-01','2023-12-31','Inflation shock and tightening'),
 ('E9','2024-01-01','2026-05-29','Recent policy divergence'),
]
GROUPS = {
 'Base': {'Commodity':['AUD','NZD','CAD'], 'European':['EUR','GBP','NOK','SEK'], 'Safe Haven/Funding':['JPY','CHF']},
 'NOK_to_Commodity': {'Commodity':['AUD','NZD','CAD','NOK'], 'European':['EUR','GBP','SEK'], 'Safe Haven/Funding':['JPY','CHF']},
 'GBP_separate': {'Commodity':['AUD','NZD','CAD'], 'European':['EUR','NOK','SEK'], 'Safe Haven/Funding':['JPY','CHF'], 'GBP (standalone)':['GBP']},
}
assert all(sorted(sum(group.values(),[]))==sorted(CURRENCIES) for group in GROUPS.values())

m=macro_monthly
vix,hy=m.vix_level,m.us_hy_oas
m['risk']='Intermediate'
m.loc[(vix>=RISK_STRESS_VIX)|(hy>=RISK_STRESS_HY),'risk']='Stress'
m.loc[(vix<RISK_CALM_VIX)&(hy<RISK_CALM_HY),'risk']='Calm'
m.loc[vix.isna()|hy.isna(),'risk']='NA'
vp,hp=m.vix_pctile_5y,m.hy_oas_pctile_5y
m['risk_pctile']='Mid'
m.loc[(vp>=.80)|(hp>=.80),'risk_pctile']='High'
m.loc[(vp<.50)&(hp<.50),'risk_pctile']='Low'
m.loc[vp.isna()|hp.isna(),'risk_pctile']='NA'
m['dxy_3m_ret']=m.dxy_level/m.dxy_level.shift(3)-1
m['wti_3m_ret']=m.wti_level/m.wti_level.shift(3)-1
m['us2y_3m_delta_pp']=m.us_2y-m.us_2y.shift(3)
m['usd_trend']=np.select([m.dxy_3m_ret>=.02,m.dxy_3m_ret<=-.02],['USD_up','USD_down'],default='USD_sideways')
m.loc[m.dxy_3m_ret.isna(),'usd_trend']='NA'
m['oil']=np.select([m.wti_3m_ret>=.10,m.wti_3m_ret<=-.10],['Oil_up','Oil_down'],default='Oil_sideways')
m.loc[m.wti_3m_ret.isna(),'oil']='NA'
m['us_rates']=np.select([m.us2y_3m_delta_pp>=.50,m.us2y_3m_delta_pp<=-.50],['US_yields_rising','US_yields_falling'],default='US_yields_stable')
m.loc[m.us2y_3m_delta_pp.isna(),'us_rates']='NA'

def era_label(day):
    s=str(day)[:10]
    return next((code for code,start,end,_ in HISTORICAL_ERAS if start<=s<=end),'NA')
m['historical_era']=[era_label(day) for day in m.date]
REGIME_DIMS = ('risk','risk_pctile','usd_trend','us_rates','oil','historical_era','technical_trend','technical_vol')
print('Historical macro categories:')
display(m[['date','vix_level','us_hy_oas','risk','risk_pctile','usd_trend','us_rates','oil']].head(10))

# 月度状态后向匹配到日频索引，无需额外 CSV 文件。
def macro_at_daily_dates(dates):
    ds=np.asarray(pd.to_datetime(dates),dtype='datetime64[ns]')
    month_ds=m.date.to_numpy(dtype='datetime64[ns]')
    j=np.searchsorted(month_ds,ds,side='right')-1
    ans={}
    for label in ['risk','risk_pctile','usd_trend','us_rates','oil']:
        v=m[label].to_numpy(dtype='U28')
        ans[label]=np.where(j>=0,v[np.maximum(j,0)],'NA')
    return ans

def technical_regime(features,prices):
    eff=np.asarray(features['eff63'])
    vol=np.asarray(features['vol21'])
    # 特别提醒：full sample 三分位阈值仅用于事后统计分组，不可回测成前瞻交易决策。
    q1,q2=np.nanquantile(eff,[1/3,2/3]);med=np.nanmedian(vol)
    trend=np.full(len(prices),'NA',dtype='<U20')
    trend[np.isfinite(eff)&(eff<=q1)]='Range'
    trend[np.isfinite(eff)&(eff>=q2)]='Trend'
    trend[np.isfinite(eff)&(eff>q1)&(eff<q2)]='Transition'
    vv=np.full(len(prices),'NA',dtype='<U20')
    vv[np.isfinite(vol)&(vol<=med)]='LowVol'
    vv[np.isfinite(vol)&(vol>med)]='HighVol'
    return {'technical_trend':trend,'technical_vol':vv}

## 4. 统一执行引擎（四种持仓方式）

令 `signal[t]` 为 t 收盘**之后**才知道的指标信号；日频的仓位变化不能使用已过去的同一收盘价成交。因此信号在 **t+1 收盘执行**，从 t+1→t+2 的日收益开始盈亏。所有方式均以每个货币对 1x 名义敞口的日简单收益逐日复合，无成本、无 carry。

In [ ]:
_CALENDAR_CACHE={}
def week_groups(dates):
    key=(len(dates),str(dates[0]),str(dates[-1]))
    if key in _CALENDAR_CACHE:return _CALENDAR_CACHE[key]
    ds=pd.DatetimeIndex(pd.to_datetime(dates))
    monday=(ds-pd.to_timedelta(ds.weekday,unit='D')).strftime('%Y-%m-%d')
    idx=[]
    for _,group in pd.Series(np.arange(len(ds))).groupby(monday,sort=False):
        g=group.to_numpy()
        idx.append((int(g[0]),int(g[-1])))
    _CALENDAR_CACHE[key]=idx
    return idx

def execute_mode(prices, dates, sig, mode):
    n=len(prices)
    executed=np.zeros(n,dtype=np.int8)
    decision=np.full(n,-1,dtype=np.int32)
    sig=np.asarray(sig,dtype=np.int8)
    weeks=week_groups(dates)
    if mode.startswith('Daily'):
        executed[1:]=sig[:-1]
        decision[2:]=np.arange(n-2)
        if mode=='Daily Weekflat':
            weekdays=pd.DatetimeIndex(dates).weekday.to_numpy()
            executed[weekdays==4]=0    # 周五收盘离场，周一价格变化不计盈亏
    elif mode=='Weekly Flat':
        for k in range(1,len(weeks)):
            start,end=weeks[k]
            last_friday=weeks[k-1][1]
            if end>start:
                executed[start:end]=sig[last_friday]
            decision[start:end+1]=last_friday
        # 每个周末最后收盘价之后都没有外币敞口
        assert all(executed[e]==0 for _,e in weeks)
    elif mode=='Fixed 5D':
        for t in range(0,n,5):
            start=t+1
            if start>=n: break
            stop=min(t+6,n)         # position close t+1 .. t+5；returns end t+2..t+6
            executed[start:stop]=sig[t]
            decision[start+1:min(t+7,n)]=t
    else:
        raise ValueError(f'Unrecognized mode: {mode}')
    held=np.zeros(n,dtype=np.int8)
    held[1:]=executed[:-1]
    underlying=np.zeros(n,dtype=float)
    underlying[1:]=prices[1:]/prices[:-1]-1.
    daily_ret=held*underlying
    equity=np.cumprod(1.+daily_ret)
    turnover=np.abs(executed.astype(int)-held.astype(int)).astype(float)
    assert np.isfinite(equity).all()
    if mode=='Weekly Flat':
        assert all(held[s]==0 for s,e in weeks)  # 周一终止的 Fri->Mon 收益区间空仓
    if mode=='Daily Weekflat':
        assert np.all(held[pd.DatetimeIndex(dates).weekday.to_numpy()==0]==0)
    return {'sig':sig,'executed':executed,'held':held,'daily_ret':daily_ret,
            'base_ret':underlying,'equity':equity,'turnover':turnover,'decision_idx':decision}

def summary_stats(run):
    r=run['daily_ret'];h=run['held'];n=len(r)
    eq=np.r_[1.,run['equity']]
    dd=eq/np.maximum.accumulate(eq)-1.
    sd=np.std(r,ddof=1)
    return dict(total_return_pct=100*(eq[-1]-1),
        annualized_return_pct=100*(eq[-1]**(252/n)-1),
        sharpe=float(np.mean(r)/sd*np.sqrt(252)) if sd>1e-12 else np.nan,
        annualized_vol_pct=float(sd*np.sqrt(252)*100),max_drawdown_pct=float(dd.min()*100),
        average_daily_bp=float(np.mean(r)*1e4),active_pct=float(100*np.mean(h!=0)),
        long_foreign_pct=float(100*np.mean(h==1)),long_usd_pct=float(100*np.mean(h==-1)),
        turnover_units=float(run['turnover'].sum()),rebalances=int(np.sum(run['turnover']>0)),
        active_day_win_rate_pct=float(100*np.mean(r[h!=0]>0)) if np.any(h!=0) else np.nan)

## 5. 一键运行：9 pairs × 20 signals × 4 execution modes

主计算将同时保存两类数据：`results[(pair,strategy,mode)]` 中的完整日频收益/仓位/净值数组，以及包含 **9 × 22 × 4 = 792** 项（包括每个 pair 的两种固定方向基准）的绩效表。全部数据在内存中产生，不读取旧 CSV。

In [ ]:
results={}
price_dates={}
prices_by_pair={}
features_by_pair={}
signals_by_pair={}
labels_by_pair={}
metrics_rows=[]

for pair in CURRENCIES:
    d=source_daily[pair]
    dates=pd.DatetimeIndex(pd.to_datetime(d['date']))
    px_arr=np.asarray(d['px'],dtype=float)
    price_dates[pair]=dates
    prices_by_pair[pair]=px_arr
    signals,features=complete_signals(px_arr)
    signals_by_pair[pair]=signals
    features_by_pair[pair]=features
    global_states=macro_at_daily_dates(dates)
    global_states.update(technical_regime(features,px_arr))
    global_states['historical_era']=np.array([era_label(x.strftime('%Y-%m-%d')) for x in dates],dtype='U4')
    labels_by_pair[pair]=global_states
    wk=week_groups(dates)
    for rule in ALL_RULES:
        for mode in MODES:
            out=execute_mode(px_arr,dates,signals[rule],mode)
            results[(pair,rule,mode)]=out
            metrics_rows.append({'pair':pair,'strategy':rule,'mode':mode,
                'family':RULES[rule][0], 'hypothesis':RULES[rule][1],
                'benchmark':rule in BENCHMARKS, **summary_stats(out)})
    print(f'{pair}: {len(px_arr):,} prices, {len(ALL_RULES)*len(MODES)} independent backtests')
metrics=pd.DataFrame(metrics_rows)
assert len(metrics)==9*22*4
assert len(results)==len(metrics)
print('FINISHED:',len(metrics),'strategy/mode/pair combinations (720 technical; 72 benchmarks).')
display(metrics.query('mode == @MAIN_MODE and benchmark == False').head(12))

### 5.1 核心执行断言与样本核对

检验日频延迟、周末归零、周度平仓、完整五日持仓的五段收益、原始 log return，以及价格终点/日期。如果这些断言失败，请不要使用后续图表。

In [ ]:
for pair in CURRENCIES:
    dates=price_dates[pair]
    n=len(dates)
    for strat in STRATEGIES:
        d=results[(pair,strat,'Daily Continuous')]
        assert np.array_equal(d['executed'][1:],d['sig'][:-1])
        assert np.array_equal(d['held'][2:],d['sig'][:-2])
        assert np.allclose(d['daily_ret'],d['held']*d['base_ret'])
        w=results[(pair,strat,'Daily Weekflat')]
        assert np.all(w['held'][dates.weekday==0]==0)
        v=results[(pair,strat,'Weekly Flat')]
        assert all(v['executed'][last]==0 for _,last in week_groups(dates))
        assert np.all(v['held'][dates.weekday==0]==0)
        h=results[(pair,strat,'Fixed 5D')]
        for t in range(0,n-6,5):
            assert np.all(h['held'][t+2:t+7]==h['sig'][t])
        for r in (d,w,v,h):
            assert np.isclose(r['equity'][-1], np.prod(1+r['daily_ret']),rtol=1e-10,atol=1e-10)
print('CHECKS PASSED: signal lags, 5D windows, weekflat positions, compound P&L, all 720 technical simulations.')

## 6. 绩效与交易持仓明细

盈亏曲线按每日 `1x` 名义仓位简单收益复合。下文生成真实执行口径的各笔交易台账：`Weekly Flat` 和 `Fixed 5D` 保存每个规则决定周期；两个日频模式则将连续相同方向的仓位视为一个 Episode。日频 Episode 的持续天数可以不同，不能把它们的单笔平均收益直接和固定周期的周收益比较。

In [ ]:
def episodic_ledger(pair,rule,mode):
    """Vectorized trade ledger, including zero-signal fixed cycles for denominators."""
    r=results[(pair,rule,mode)]
    dt=price_dates[pair]
    days=dt.strftime('%Y-%m-%d').to_numpy()
    prices=prices_by_pair[pair]
    n=len(days);sig=r['sig'];exe=r['executed'];underlying=r['base_ret']
    labels=labels_by_pair[pair]
    if mode=='Weekly Flat':
        weeks=week_groups(dt)
        dec=np.array([weeks[k-1][1] for k in range(1,len(weeks))],dtype=int)
        a=np.array([weeks[k][0] for k in range(1,len(weeks))],dtype=int)
        z=np.array([weeks[k][1] for k in range(1,len(weeks))],dtype=int)
        pos=sig[dec]
        kind='weekly_cycle'
    elif mode=='Fixed 5D':
        dec=np.arange(0,n-1,5,dtype=int)
        a=dec+1
        z=np.minimum(dec+6,n-1)
        pos=sig[dec]
        kind='fixed5_cycle'
    else:
        a=np.flatnonzero((exe!=0)&(exe!=np.r_[0,exe[:-1]])).astype(int)
        bounds=np.r_[np.flatnonzero(exe[1:]!=exe[:-1])+1,n]
        stops=bounds[np.searchsorted(bounds,a,side='right')]
        z=np.minimum(stops,n-1)
        dec=a-1
        pos=exe[a]
        kind='daily_episode'
    valid=z>a
    dec,a,z,pos=dec[valid],a[valid],z[valid],pos[valid]
    if not len(a):return pd.DataFrame()
    # Prefix sum of log(1 - spot daily return) reproduces a daily rebalanced short-FX strategy.
    neg_prefix=np.r_[0,np.cumsum(np.log1p(-underlying[1:]))]
    def at_horizon(h):
        last=np.minimum(a+h,z)
        fwd_long=(prices[last]/prices[a]-1)
        fwd_short=np.expm1(neg_prefix[last]-neg_prefix[a])
        pnl=np.where(pos==1,fwd_long,np.where(pos==-1,fwd_short,0.))
        return np.where(z>=a+h,pnl*1e4,np.nan)
    final_long=prices[z]/prices[a]-1
    final_short=np.expm1(neg_prefix[z]-neg_prefix[a])
    final=np.where(pos==1,final_long,np.where(pos==-1,final_short,0.))
    # Count adverse *daily return intervals* using cumsum for each direction.
    fxbad=np.r_[0,np.cumsum(underlying[1:]<0)]
    usdbad=np.r_[0,np.cumsum(underlying[1:]>0)]
    bad_days=np.where(pos==1,fxbad[z]-fxbad[a],np.where(pos==-1,usdbad[z]-usdbad[a],0))
    lin_prefix=np.r_[0,np.cumsum(underlying[1:])]
    linear=np.where(pos==0,0.,pos*(lin_prefix[z]-lin_prefix[a])*1e4)
    out=pd.DataFrame({
      'pair':pair,'strategy':rule,'mode':mode,'trade_type':kind,
      'decision_date':days[np.clip(dec,0,n-1)],
      'entry_date':days[a],'exit_date':days[z],
      'position':pos.astype(int),'holding_intervals':(z-a).astype(int),
      'fully_observed':z<(n-1),
      'return_bp':final*1e4,
      'base_spot_change_bp':final_long*1e4,
      'risk':labels['risk'][np.clip(dec,0,n-1)],
      'technical_trend':labels['technical_trend'][np.clip(dec,0,n-1)],
      'technical_vol':labels['technical_vol'][np.clip(dec,0,n-1)],
      'historical_era':labels['historical_era'][np.clip(dec,0,n-1)],
      'year':dt.year.to_numpy()[a],
      'long_fx_contribution_bp':np.where(pos==1,linear,0.),
      'long_usd_contribution_bp':np.where(pos==-1,linear,0.),
      'bad_days':bad_days.astype(int),
    })
    for i in range(1,6):out[f'path_day_{i}_bp']=at_horizon(i)
    return out

trade_chunks=[]
for mode in MODES:
    for pair in CURRENCIES:
        for rule in STRATEGIES:
            trade_chunks.append(episodic_ledger(pair,rule,mode))
trades=pd.concat(trade_chunks,ignore_index=True)
print('Trade records:',len(trades),'| full five-day cycles:',int(((trades['mode']=='Fixed 5D')&(trades['holding_intervals']==5)).sum()))
display(trades.groupby(['mode']).agg(trades=('position','size'),active=('position',lambda x:(x!=0).sum()),
                                       median_intervals=('holding_intervals','median')).reset_index())

### 6.1 Annual P&L、历史时期及持仓时长

统计是独立货币对的策略表现，**不要**将组内收益直接相加视为投资组合。

In [ ]:
annual_rows=[]
period_rows=[]
PERIODS=[('2000–2012','2000-01-01','2012-12-31'),('2013–2026','2013-01-01','2026-12-31'),
         ('2000–2007','2000-01-01','2007-12-31'),('2008–2014','2008-01-01','2014-12-31'),
         ('2015–2019','2015-01-01','2019-12-31'),('2020–2026','2020-01-01','2026-12-31')]
for (pair,rule,mode),r in results.items():
    dates=price_dates[pair];ret=r['daily_ret'];h=r['held']
    for year in sorted(set(dates.year)):
        mask=dates.year==year;x=ret[mask]
        annual_rows.append({'pair':pair,'strategy':rule,'mode':mode,'year':int(year),
                            'return_pct':100*(np.prod(1+x)-1),'active_days':int(np.sum(h[mask]!=0))})
    for name,first,last in PERIODS:
        mask=(dates>=first)&(dates<=last);x=ret[mask]
        period_rows.append({'pair':pair,'strategy':rule,'mode':mode,'period':name,
                            'return_pct':100*(np.prod(1+x)-1),'mean_daily_bp':1e4*x.mean()})
annual=pd.DataFrame(annual_rows)
historical_periods=pd.DataFrame(period_rows)
completed=trades.loc[(trades.position!=0)&trades.fully_observed].copy()
holding=completed.groupby(['pair','strategy','mode'],as_index=False).agg(
    n_episodes=('position','size'),mean_holding_intervals=('holding_intervals','mean'),
    median_holding_intervals=('holding_intervals','median'),p90_holding_intervals=('holding_intervals',lambda x:np.percentile(x,90)),
    win_rate_pct=('return_bp',lambda x:100*(x>0).mean()),
    mean_episode_bp=('return_bp','mean'))
display(holding.query("mode == 'Daily Continuous'").sort_values('mean_holding_intervals',ascending=False).head(10))

## 7. Macro / Technical Regime 条件收益与交易机制

为了清晰比较执行频率，**按日条件统计**首先比较所有模式的单位暴露交易日收益（bp/active day），并展示包含 Flat 在内的均值；固定交易期另外用交易台账计算 bp/active trade。两种度量单位不能混淆。

In [ ]:
conditional=[]
for (pair,rule,mode),r in results.items():
    n=len(price_dates[pair]);ret=r['daily_ret'];h=r['held'];idx=r['decision_idx']
    label_source=labels_by_pair[pair]
    for dimension in REGIME_DIMS:
        source=np.asarray(label_source[dimension])
        use=np.where(idx>=0,source[np.clip(idx,0,n-1)],'NA')
        for state in np.unique(use):
            if state=='NA':continue
            mask=use==state
            active=mask&(h!=0)
            vals=ret[active];all_vals=ret[mask]
            conditional.append({'pair':pair,'strategy':rule,'mode':mode,'family':RULES[rule][0],
              'hypothesis':RULES[rule][1], 'dimension':dimension,'state':state,
              'n_sessions':int(mask.sum()),'active_sessions':int(active.sum()),
              'mean_all_day_bp':float(all_vals.mean()*1e4),
              'mean_active_day_bp':float(vals.mean()*1e4) if len(vals) else np.nan,
              'active_win_pct':float(100*(vals>0).mean()) if len(vals) else np.nan,
              'active_worst_decile_bp':float(1e4*np.mean(np.sort(vals)[:max(1,int(np.ceil(.1*len(vals))))])) if len(vals)>=10 else np.nan,
              'long_fx_contrib_linear_bp':float(1e4*ret[mask&(h==1)].sum()),
              'long_usd_contrib_linear_bp':float(1e4*ret[mask&(h==-1)].sum())})
conditional=pd.DataFrame(conditional)

# 第一层：独立 Pair 的 Stress – Calm 条件活跃日收益差
v=conditional.query("dimension == 'risk'")
s=v.pivot_table(index=['pair','strategy','mode'],columns='state',
                values='mean_active_day_bp').reset_index()
s['stress_minus_calm_bp_day']=s['Stress']-s['Calm']
risk_contrast=s.dropna(subset=['stress_minus_calm_bp_day'])

# 第二层：每组先分别计算各 Pair 的条件收益差，再取中位数；不是 Portfolio P&L
def grouped_contrasts(mode=MAIN_MODE,scenario=GROUP_SCENARIO):
    mapping=GROUPS[scenario]
    rows=[]
    for group,pairs in mapping.items():
        for rule in STRATEGIES:
            sub=risk_contrast[(risk_contrast['mode']==mode)&(risk_contrast['strategy']==rule)&risk_contrast.pair.isin(pairs)]
            values=sub['stress_minus_calm_bp_day'].dropna().to_numpy()
            rows.append({'mode':mode,'scenario':scenario,'group':group,'strategy':rule,
             'family':RULES[rule][0], 'n_pair':len(values),
             'n_pair_stress_better':int((values>0).sum()),
             'median_stress_minus_calm_bp_day':float(np.median(values)) if len(values) else np.nan})
    return pd.DataFrame(rows)

group_table=pd.concat([grouped_contrasts(mode,scenario) for mode in MODES for scenario in GROUPS],ignore_index=True)
display(grouped_contrasts().pivot(index='strategy',columns='group',
                                  values='median_stress_minus_calm_bp_day').round(2))

### 7.1 固定五日、周度交易的宏观状态收益（bp / 活跃交易）

这里每条完整持仓周期算一次观测，避免将一个五日交易重复计数五次；未完成的末尾持仓周期剔除。对于日频持续持仓，优先参考上面的 active-day 归因，或者单独查看 `holding` 的 Episode 持续时间。

In [ ]:
fixed_trade = trades[(trades['mode']=='Fixed 5D') & (trades.holding_intervals==5)]
weekly_trade = trades[trades['mode']=='Weekly Flat']
trade_conditional_parts=[]
for mode,data in [('Fixed 5D',fixed_trade),('Weekly Flat',weekly_trade)]:
    for dimension in ('risk','historical_era'):
        keys=['pair','strategy',dimension]
        d=data[data[dimension]!='NA']
        all_values=d.groupby(keys,observed=True)['return_bp'].agg(cycles='size',mean_all_cycle_bp='mean')
        active=d[d.position!=0]
        active_values=active.groupby(keys,observed=True)['return_bp'].agg(
            active_cycles='size',mean_active_cycle_bp='mean',
            active_win_pct=lambda x:float(100*np.mean(x>0)),
            worst_decile_bp=lambda x:float(x.nsmallest(max(1,int(np.ceil(len(x)*.1)))).mean()) if len(x)>=10 else np.nan)
        joined=all_values.join(active_values).reset_index().rename(columns={dimension:'state'})
        joined.insert(0,'dimension',dimension)
        joined.insert(0,'mode',mode)
        trade_conditional_parts.append(joined)
trade_conditional=pd.concat(trade_conditional_parts,ignore_index=True)
display(trade_conditional.query("mode == 'Weekly Flat' and pair == 'JPY' and strategy == 'MA_20_60' and dimension == 'risk'"))

## 8. 持仓期间价格路径、极端亏损、共振事件

对每个货币对与策略，在**完全持有五个日收益区间**的交易里取最低 10% 的交易作为 worst decile，再看每笔交易的第 1–5 日累计收益和持续不利价格变化。分位数划分使用同一全历史样本，因此是事后诊断，不能拿来设置事前止损阈值。

In [ ]:
five_active=trades[(trades['mode']=='Fixed 5D') & (trades.position!=0) & (trades.holding_intervals==5)].copy()
cut=five_active.groupby(['pair','strategy'],observed=True)['return_bp'].transform('quantile',q=.10)
five_active['worst_decile']=five_active['return_bp']<=cut
path_cols=[f'path_day_{i}_bp' for i in range(1,6)]
tail_grp=five_active.groupby(['pair','strategy','risk'],observed=True)
tail_stats=tail_grp['return_bp'].agg(n='size',mean_trade_bp='mean',median_trade_bp='median',
    fraction_negative_pct=lambda x:float(100*(x<0).mean()),
    left_tail10_mean_bp=lambda x:float(x.nsmallest(max(1,int(np.ceil(len(x)*.1)))).mean())).reset_index()
frequency=five_active.groupby(['pair','strategy','risk'],observed=True)['worst_decile'].mean().mul(100).reset_index(name='worst_decile_frequency_pct')
tail_stats=tail_stats.merge(frequency,on=['pair','strategy','risk'])
bad=five_active[five_active.worst_decile]
loss_path=bad.groupby(['strategy','risk'],as_index=False,observed=True)[path_cols].mean()
worst_cases=five_active.nsmallest(100,'return_bp')
print('Completed active 5D trade windows:',len(five_active),'worst decile records:',len(bad))
display(tail_stats.query("strategy == 'BB_MR_20'").head(10))

# 同步亏损事件，使用向量化聚合，避免成千上万次 Python lambda
sync=(five_active.groupby(['strategy','decision_date'],observed=True)
  .agg(pair_count=('pair','size'),tail_losses=('worst_decile','sum'),
       min_pos=('position','min'),max_pos=('position','max'),mean_pnl_bp=('return_bp','mean'))
  .reset_index())
sync['all_long_fx']=(sync.min_pos==1)&(sync.max_pos==1)
sync['all_long_usd']=(sync.min_pos==-1)&(sync.max_pos==-1)
common_tail_events=sync[(sync.pair_count>=7)&(sync.tail_losses>=6)].sort_values(
    ['tail_losses','mean_pnl_bp'],ascending=[False,True])
display(common_tail_events.head(12))

## 9. 跨时期稳定性与同步年度区块 Bootstrap

对 **事先固定**的六套代表规则进行同步整年重抽样；九个货币对同一轮使用相同的年份序列，尽量保留美元共同冲击。这里显示的是 **组内 Pair 级别 Stress − Calm 差异的中位数**，不构建组合。Bootstrap 置信区间为探索性的，并未对大量策略/分组选择做多重检验校正。

In [ ]:
ANCHOR_RULES=['RSI_MR_14','RSI_MOM_14','MA_20_60','BB_MR_20','BB_BO_20','DONCHIAN_BO_20']

def half_period_contrast(mode=MAIN_MODE):
    rows=[]
    for pair in CURRENCIES:
      dates=price_dates[pair]
      for rule in STRATEGIES:
        r=results[(pair,rule,mode)]
        ids=r['decision_idx'];lab=labels_by_pair[pair]['risk']
        selected=np.where(ids>=0,lab[np.clip(ids,0,len(dates)-1)],'NA')
        for label,start,end in [('Early','2000-01-01','2012-12-31'),('Late','2013-01-01','2026-12-31')]:
          mask=(dates>=start)&(dates<=end)&(r['held']!=0)
          stress=r['daily_ret'][mask&(selected=='Stress')]
          calm=r['daily_ret'][mask&(selected=='Calm')]
          rows.append({'pair':pair,'strategy':rule,'mode':mode,'half':label,
             'n_stress':len(stress),'n_calm':len(calm),
             'stress_minus_calm_bp_day':1e4*(stress.mean()-calm.mean()) if len(stress)>0 and len(calm)>0 else np.nan})
    return pd.DataFrame(rows)

stability=pd.concat([half_period_contrast(mode) for mode in MODES],ignore_index=True)

# 每年每对币的条件收益总量+次数，用于重采样时保持同一年所有币种共同冲击
BOOT_YEARS=np.arange(2000,2027)

def bootstrap_group(mode='Weekly Flat', scenario='Base', rules=ANCHOR_RULES, draws=BOOTSTRAP_DRAWS, seed=RANDOM_SEED):
    years=BOOT_YEARS
    arrays=np.zeros((len(CURRENCIES),len(rules),len(years),2,2),dtype=float)
    for pi,pair in enumerate(CURRENCIES):
      dates=price_dates[pair]
      for ri,rule in enumerate(rules):
        r=results[(pair,rule,mode)]
        ix=r['decision_idx'];labels=labels_by_pair[pair]['risk']
        label=np.where(ix>=0,labels[np.clip(ix,0,len(dates)-1)],'NA')
        for yi,y in enumerate(years):
          for gi,g in enumerate(['Stress','Calm']):
            mask=(dates.year==y)&(label==g)&(r['held']!=0)
            arrays[pi,ri,yi,gi,0]=r['daily_ret'][mask].sum()*1e4
            arrays[pi,ri,yi,gi,1]=mask.sum()
    rng=np.random.default_rng(seed)
    sampled=rng.integers(len(years),size=(draws,len(years)))
    group_rows=[]
    for group,pairs in GROUPS[scenario].items():
      sel=[CURRENCIES.index(p) for p in pairs]
      for ri,rule in enumerate(rules):
        base=arrays[sel,ri].sum(axis=1)
        if np.any(base[:,:,1]==0):continue
        point=np.median(base[:,0,0]/base[:,0,1]-base[:,1,0]/base[:,1,1])
        reps=[]
        for years_idx in sampled:
          v=arrays[sel,ri][:,years_idx,:,:].sum(axis=1)
          if np.any(v[:,:,1]==0):continue
          reps.append(np.median(v[:,0,0]/v[:,0,1]-v[:,1,0]/v[:,1,1]))
        lo,hi=np.percentile(reps,[2.5,97.5]) if len(reps)>=max(100,draws//3) else (np.nan,np.nan)
        group_rows.append({'mode':mode,'scenario':scenario,'group':group,'strategy':rule,
           'n_pairs':len(pairs),'estimate_bp_active_day':point,'ci_low':lo,'ci_high':hi,
           'draws_used':len(reps),'contains_zero':bool(lo<=0<=hi) if np.isfinite(lo) else None})
    return pd.DataFrame(group_rows)

bootstrap=bootstrap_group(mode=MAIN_MODE)
display(bootstrap.round(3))

## 10. 可视化一：全部 180 套策略的交互式累计 P&L

`plot_all_180_curves` 提供九个货币对的下拉菜单；每个币种包含 20 套技术策略（可单独点击图例隐藏、缩放、悬停查看）。`plot_pair_modes` 查看单一 Pair × Strategy 在四种执行方式下的差异，并展示日频实际持仓。**完整日频数据**来自 `results`，无需单独下载其他文件。

In [ ]:
def curve_points(pair,run,sample='month_end'):
    dates=price_dates[pair]
    if sample=='daily':return dates,run['equity']
    periods=dates.to_period('M')
    indices=np.r_[np.flatnonzero(periods[1:]!=periods[:-1]),len(dates)-1]
    return dates[indices],run['equity'][indices]

def plot_all_180_curves(mode=MAIN_MODE,sample=PLOT_SAMPLE):
    fig=go.Figure()
    for pair in CURRENCIES:
      for rule in STRATEGIES:
        dates,eq=curve_points(pair,results[(pair,rule,mode)],sample)
        fig.add_trace(go.Scatter(x=dates,y=100*(eq-1),name=rule,mode='lines',
               line=dict(width=1.3),visible=pair==CURRENCIES[0],
               hovertemplate='%{x|%Y-%m-%d}<br>P&L %{y:.2f}%<extra>'+rule+'</extra>'))
    n=len(STRATEGIES)
    buttons=[]
    for idx,pair in enumerate(CURRENCIES):
      on=[i//n==idx for i in range(len(fig.data))]
      buttons.append(dict(label=pair,method='update',args=[{'visible':on},{'title':f'{pair} vs USD | {mode} | 20 technical strategies'}]))
    fig.update_layout(title=f'{CURRENCIES[0]} vs USD | {mode} | 20 technical strategies',
       template='plotly_white',height=620,hovermode='x unified',
       updatemenus=[dict(buttons=buttons,active=0,x=0.01,y=1.19,showactive=True)],
       legend=dict(orientation='h',y=-.22),margin=dict(t=120,b=140),
       xaxis_title='Date',yaxis_title='Cumulative P&L (%) — not a portfolio')
    return fig

def plot_pair_modes(pair='EUR',strategy='RSI_MR_14',sample='daily'):
    fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.08,
                      row_heights=[.75,.25],subplot_titles=['Cumulative P&L (%)','Daily Continuous: held position'])
    for mode in MODES:
      data=results[(pair,strategy,mode)]
      dates,eq=curve_points(pair,data,sample)
      fig.add_trace(go.Scatter(x=dates,y=(eq-1)*100,name=mode,mode='lines'),row=1,col=1)
    r=results[(pair,strategy,'Daily Continuous')]
    fig.add_trace(go.Scatter(x=price_dates[pair],y=r['held'],mode='lines',
                             name='Held position',line=dict(color='gray',width=1)),row=2,col=1)
    fig.update_yaxes(title_text='P&L (%)',row=1,col=1)
    fig.update_yaxes(title_text='FX / Flat / USD',tickvals=[-1,0,1],row=2,col=1)
    fig.update_layout(title=f'{pair} | {strategy} | four execution methods',
                      template='plotly_white',height=710,hovermode='x unified')
    return fig

fig_180=plot_all_180_curves()
fig_180.show()
# 单笔比较，可以修改下面任意两个选项：
plot_pair_modes(pair='JPY',strategy='MA_20_60',sample='month_end').show()

## 11. 可视化二：技术策略、货币组与 Regime 的横向比较

左侧热图呈现单个 pair 的完整历史 P&L；右侧或下方热图呈现 **组内 Pair 的 Stress − Calm 条件活跃日收益中位数（bp）**。前者是历史收益，不是预测；后者是 Regime 的描述性相关性，不是 Group Portfolio 收益。

In [ ]:
def plot_pair_performance(mode=MAIN_MODE,measure='total_return_pct'):
    d=metrics[(metrics['mode']==mode)&(~metrics.benchmark)]
    grid=d.pivot(index='strategy',columns='pair',values=measure).reindex(index=STRATEGIES,columns=CURRENCIES)
    fig=px.imshow(grid,aspect='auto',color_continuous_scale='RdBu',color_continuous_midpoint=0,
         labels={'x':'Currency','y':'Technical rule','color':measure},
         title=f'FX Technical Strategies | {mode} | {measure}')
    fig.update_layout(height=820,template='plotly_white')
    return fig

def plot_group_regime(mode=MAIN_MODE,scenario=GROUP_SCENARIO):
    d=grouped_contrasts(mode,scenario)
    group_order=list(GROUPS[scenario])
    table=d.pivot(index='strategy',columns='group',values='median_stress_minus_calm_bp_day').reindex(
         index=STRATEGIES,columns=group_order)
    fig=px.imshow(table,aspect='auto',color_continuous_scale='RdBu',color_continuous_midpoint=0,
         labels={'x':'Economic currency category','y':'Technical rule','color':'Stress - Calm (bp/active day)'},
         title=f'Currency grouping: {scenario} | {mode} | Regime contrast')
    fig.update_layout(height=800,template='plotly_white')
    return fig

plot_pair_performance().show()
plot_group_regime().show()

## 12. 可视化三：年度与历史阶段 / 宏观条件下的策略表现

宏观分期来自研究者基于经济史设定的 `E1...E9` 标签。**不能**当作发行时点已知的 Regime 预测信号。可以切换货币组、交易方式以及用于比较的策略。

In [ ]:
def plot_annual(pair='AUD',rule='BB_MR_20'):
    d=annual[(annual.pair==pair)&(annual.strategy==rule)]
    fig=px.bar(d,x='year',y='return_pct',color='mode',barmode='group',
       title=f'Annual P&L by execution method | {pair} {rule}',
       labels={'return_pct':'Annual compounded return (%)'})
    fig.update_layout(template='plotly_white',height=480)
    return fig

def plot_historical_eras(pair='JPY',rule='MA_20_60',mode=MAIN_MODE):
    r=results[(pair,rule,mode)]
    idx=r['decision_idx'];labs=labels_by_pair[pair]['historical_era'];n=len(idx)
    states=np.where(idx>=0,labs[np.clip(idx,0,n-1)],'NA')
    values=[]
    for eid,a,b,title in HISTORICAL_ERAS:
      mask=states==eid
      vals=r['daily_ret'][mask]
      values.append({'historical_era':eid,'label':title,'all_day_mean_bp':float(vals.mean()*1e4) if len(vals) else np.nan,
                     'linear_sum_bp':float(vals.sum()*1e4), 'n_sessions':len(vals)})
    out=pd.DataFrame(values)
    fig=px.bar(out,x='historical_era',y='all_day_mean_bp',hover_name='label',
         title=f'Historical macro eras: conditional daily mean P&L | {pair} {rule} ({mode})',
         labels={'all_day_mean_bp':'Average daily P&L (bp, all covered sessions)'})
    fig.update_layout(template='plotly_white',height=450)
    return out,fig

plot_annual().show()
era_table,era_chart=plot_historical_eras()
era_chart.show()

## 13. 可视化四：极端损失形成过程与实际持仓

对固定五日交易的 Worst-Decile Trades，绘制日 1–5 的**平均累计收益路径**。注意这是按最终结果事后筛选的交易，不能把前几天的平均轨迹当作可部署止损策略。

In [ ]:
def plot_worst_loss_paths(strategy='BB_MR_20',pair=None):
    subset=bad[bad.strategy==strategy]
    if pair:subset=subset[subset.pair==pair]
    fig=go.Figure()
    for label in ['Stress','Calm','Intermediate']:
      part=subset[subset.risk==label]
      if not len(part):continue
      y=part[path_cols].mean().to_numpy(dtype=float)
      fig.add_trace(go.Scatter(x=[1,2,3,4,5],y=y,name=f'{label} (n={len(part)})',mode='lines+markers'))
    fig.update_layout(title=f'Worst 10% complete 5-day trades | {strategy} | {pair or "9 currencies (descriptive)"}',
        xaxis_title='Day of five-day position',yaxis_title='Mean cumulative trade return (bp)',
        template='plotly_white',height=460)
    return fig

def plot_trade_duration(pair='EUR'):
    d=completed[(completed.pair==pair)&(completed['mode'].str.startswith('Daily'))]
    fig=px.box(d,x='strategy',y='holding_intervals',color='mode',
      title=f'Variable holding durations | {pair}',labels={'holding_intervals':'Intervals held'})
    fig.update_layout(template='plotly_white',height=500,xaxis_tickangle=-50)
    return fig

plot_worst_loss_paths('BB_MR_20').show()
plot_trade_duration('EUR').show()

## 14. 保存为 CSV、Excel 兼容表与离线 HTML

以下代码导出**本次重新运行所产生**的研究结果：所有策略绩效、年度 P&L、条件收益、分组敏感性、极端事件；逐笔交易已在 `trades` 内存表，完整导出需设置 `EXPORT_ALL_TRADES = True`。交互图保存为内嵌 Plotly 的 HTML（断网亦可打开）。这些文件由 Notebook 运行后创建，原 ZIP 依然只有 Notebook + 原始 Dataset。

In [ ]:
export_tables={
 'strategy_metrics_all_modes':metrics,
 'strategy_rules':specifications,
 'data_quality_audit':source_audit,
 'macro_snapshots_rebuilt':macro_monthly,
 'annual_returns':annual,
 'historical_period_returns':historical_periods,
 'conditional_regime_active_day':conditional,
 'risk_stress_minus_calm_pairwise':risk_contrast,
 'currency_group_sensitivity':group_table,
 'trade_level_conditional_fixed_horizon':trade_conditional,
 'holding_episodes':holding,
 'tail_risk_fixed5':tail_stats,
 'worst5day_cases':worst_cases,
 'synchronous_tail_events':common_tail_events,
 'historical_stability':stability,
 'year_block_bootstrap':bootstrap,
}
for filename,frame in export_tables.items():
    frame.to_csv(OUTPUT_DIR/f'{filename}.csv',index=False,encoding='utf-8-sig')

# 月末高密度压缩曲线：9 × 22 × 4；完整每日数组可直接调用 results[(pair,strategy,mode)]。
curve_rows=[]
for (pair,rule,mode),run in results.items():
    ds,wealth=curve_points(pair,run,'month_end')
    for day,v in zip(ds,wealth):
        curve_rows.append({'pair':pair,'strategy':rule,'mode':mode,
                           'date':day.strftime('%Y-%m-%d'),'equity':v,'cumulative_pct':100*(v-1)})
monthly_curves=pd.DataFrame(curve_rows)
monthly_curves.to_csv(OUTPUT_DIR/'monthly_cumulative_pnl_all_modes.csv.gz',index=False,compression='gzip')

# 单个币种 / 指标的完整日频曲线、信号和实际持仓：可修改参数后调用。
def export_daily_trace(pair='EUR',rule='MA_20_60',mode=MAIN_MODE):
    run=results[(pair,rule,mode)]
    frame=pd.DataFrame({'date':price_dates[pair].strftime('%Y-%m-%d'),
       'close_USD_per_foreign':prices_by_pair[pair],
       'signal_at_close':run['sig'],'executed_post_close':run['executed'],
       'held_for_ending_interval':run['held'],'spot_return':run['base_ret'],
       'strategy_return':run['daily_ret'],'cumulative_pnl_pct':100*(run['equity']-1)})
    target=OUTPUT_DIR/f'daily_trace_{pair}_{rule}_{mode.replace(" ","_")}.csv'
    frame.to_csv(target,index=False)
    print('Trace saved:',target)
    return frame

export_daily_trace()
fig_180.write_html(OUTPUT_DIR/'ALL_180_Interactive_PnL.html',include_plotlyjs=True,full_html=True)
plot_pair_performance().write_html(OUTPUT_DIR/'Strategy_Return_Heatmap.html',include_plotlyjs=True)
plot_group_regime().write_html(OUTPUT_DIR/'Currency_Group_Macro_Regime.html',include_plotlyjs=True)
# 大型交易台账始终可在内存中使用。需要持久化全部 750k+ 记录时再开启：
EXPORT_ALL_TRADES = False
if EXPORT_ALL_TRADES:
    trades.to_csv(OUTPUT_DIR/'trades_all_modes.csv.gz',index=False,compression='gzip')
print('Exported',len(export_tables),'analysis tables + compressed monthly curves + 3 offline interactive HTML charts.')
print('Output folder:',OUTPUT_DIR.resolve())

## 15. 一页研究结论与后续合作边界

这是一套可复现的 **technical-rule conditional payoff framework**，而不是一套已经 OOS 验证的自动交易策略。

- **Technical Team**：解释 20 套规则在四种执行方式下的表现、趋势/反转假设、货币类别、实际 Long FX / Long USD 暴露、持仓期限、Regime 条件收益、异常亏损与历史稳定性。
- **Macro Team**：负责宏观因子选择、Regime 定义/预测和政策经济机制。Notebook 中的宏观状态是**事后归因标签**，绝不使用未来已知的宏观状态决定历史交易仓位。
- **方法论限制**：全历史事后统计、未纳入交易费用/FX swap/carry、未经实时 vintage 检验、原始文件只有工作日 Close（非统一可成交 Bid/Ask）；若将来提出实际 Regime Switching 策略，必须另外进行前瞻的时序测试和样本外验证。

**如何复现：** 两个文件放同一目录 → 安装依赖 → 打开 `.ipynb` → 选择 **Run All** → 查看交互式输出和 `outputs/`。无需其他资料。指标规则均显式写在“第 2 节”，可以独立修改并重新运行。图表、CSV 都由本 Notebook 当场计算。

**参考资料（方法背景，不代表原始数据来源）：**

- Menkhoff & Taylor (2007), *The Obstinate Passion of Foreign Exchange Professionals: Technical Analysis*, [DOI](https://doi.org/10.1257/jel.45.4.936)
- Neely, Weller & Ulrich (2009), *The Adaptive Markets Hypothesis: Evidence from the Foreign Exchange Market*, [DOI](https://doi.org/10.1017/S0022109009090245)
- Brunnermeier, Nagel & Pedersen (2009), *Carry Trades and Currency Crashes*, [NBER](https://www.nber.org/papers/w14473)
- Lustig, Roussanov & Verdelhan (2011), *Common Risk Factors in Currency Markets*, [NBER](https://www.nber.org/papers/w14082)